In [19]:
from konlpy.tag import Mecab
mecab = Mecab()
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Input, Flatten, Embedding
from tensorflow.keras.utils import to_categorical
import re
import tensorflow as tf
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, GlobalMaxPooling1D
from attention import Attention
from tensorflow.keras.layers import GlobalAveragePooling1D, LayerNormalization, MultiHeadAttention, Add
from tensorflow.keras.models import Model

In [20]:
test_data = pd.read_csv("../05machine_learning/data/bank_app_reviews_test.csv")
test_data

,리뷰일,평점,사용자리뷰,업체답변,은행명
0,2024-02-08,5,고경민계장님감사해요,"안녕하세요 최순녀 고객님. 칭찬 진심으로 감사드리며, 더욱 편리하고 안정적인 서비스...",우리
1,2023-07-24,5,저축목표피드 새로 생긴거 너무좋은데 분명 카테고리를 저축으로 했는데 왜 인식이 안되...,"신아​ 님, 안녕하세요? 뱅크샐러드 고객감동팀​입니다. 소중한 시간내어 고객센터에 ...",뱅크샐러드
2,2023-09-25,1,아니 이딴걸 편리하게 사용하는앱이라고 쳐만들엇나 이렇게 불편하게만든건 일부러그런거에...,안녕하세요. 우리은행입니다. 먼저 우리WON뱅킹 이용에 불편을 드려 죄송합니다. 보...,우리
3,2024-02-15,3,몇 년째 만족하며 사용중이라 조금식 개선되어거는 모습에 만족하며 사용중입니다. 하지...,안녕하세요? 뱅크샐러드 고객감동팀입니다. 뱅크샐러드에 KB pay를 연결해 모든 자...,뱅크샐러드
4,2023-06-19,5,스타뱅킹을 사용 하고나서부터 편안해서 좋아요,"한송림 고객님, 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 앞...",국민
...,...,...,...,...,...
9529,2025-04-05,1,만보기 이벤트는 실망스러워요. 후기 말투 다 똑같고 사기 맞죠? 양심이 참... 정...,"안녕하세요. 송송님, 토스팀입니다. 만족스러운 서비스를 제공하기 위해 노력하였으나,...",토스
9530,2023-05-17,5,기능이 많아 다 사용해보진 못 했지만 대체적으로 편한거 같아요,이강욱 고객님 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 고객...,국민
9531,2023-07-05,5,편리하네요.,"농사꾼 고객님, 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 앞...",국민
9532,2024-12-20,5,사용하기 편리해요,고객님 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. KB스타뱅킹...,국민


In [21]:
def clean_text(text):
    cleaned = re.sub(r'[^가-힣a-zA-Z0-9\s]','',text)  # 한글 영문 숫자 추출
    cleaned = re.sub(r'\s+',' ', cleaned) # 연속된 공백을 하나의 공백으로 줄임
    return cleaned.strip()

In [22]:
test_data['사용자리뷰'] = test_data['사용자리뷰'].apply(clean_text)
test_data

,리뷰일,평점,사용자리뷰,업체답변,은행명
0,2024-02-08,5,고경민계장님감사해요,"안녕하세요 최순녀 고객님. 칭찬 진심으로 감사드리며, 더욱 편리하고 안정적인 서비스...",우리
1,2023-07-24,5,저축목표피드 새로 생긴거 너무좋은데 분명 카테고리를 저축으로 했는데 왜 인식이 안되...,"신아​ 님, 안녕하세요? 뱅크샐러드 고객감동팀​입니다. 소중한 시간내어 고객센터에 ...",뱅크샐러드
2,2023-09-25,1,아니 이딴걸 편리하게 사용하는앱이라고 쳐만들엇나 이렇게 불편하게만든건 일부러그런거에요,안녕하세요. 우리은행입니다. 먼저 우리WON뱅킹 이용에 불편을 드려 죄송합니다. 보...,우리
3,2024-02-15,3,몇 년째 만족하며 사용중이라 조금식 개선되어거는 모습에 만족하며 사용중입니다 하지만...,안녕하세요? 뱅크샐러드 고객감동팀입니다. 뱅크샐러드에 KB pay를 연결해 모든 자...,뱅크샐러드
4,2023-06-19,5,스타뱅킹을 사용 하고나서부터 편안해서 좋아요,"한송림 고객님, 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 앞...",국민
...,...,...,...,...,...
9529,2025-04-05,1,만보기 이벤트는 실망스러워요 후기 말투 다 똑같고 사기 맞죠 양심이 참 정직하게 확...,"안녕하세요. 송송님, 토스팀입니다. 만족스러운 서비스를 제공하기 위해 노력하였으나,...",토스
9530,2023-05-17,5,기능이 많아 다 사용해보진 못 했지만 대체적으로 편한거 같아요,이강욱 고객님 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 고객...,국민
9531,2023-07-05,5,편리하네요,"농사꾼 고객님, 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. 앞...",국민
9532,2024-12-20,5,사용하기 편리해요,고객님 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. KB스타뱅킹...,국민


In [23]:
test_data['is_good'] = test_data['평점'].apply(lambda x: 1 if x >= 4 else 0)
test_data['is_good']

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

In [24]:
tokenized_docs = test_data['사용자리뷰'].apply(mecab.morphs)
tokenized_docs[2]

['아니',
 '이딴',
 '걸',
 '편리',
 '하',
 '게',
 '사용',
 '하',
 '는',
 '앱',
 '이',
 '라고',
 '쳐',
 '만들',
 '엇',
 '나',
 '이렇게',
 '불편',
 '하',
 '게',
 '만든',
 '건',
 '일부러',
 '그런',
 '거',
 '에요']

# train에서 사용했던 tokenizer를 불러와서 one hot encoding

In [25]:
import joblib

In [26]:
token = joblib.load("./model/bank_app_tokeizer.joblib")

In [27]:
x = token.texts_to_sequences(tokenized_docs)
print(x[0])

[6248, 327, 111, 71]


# train에서 사용했던 패딩 길이 (모델에 넣을 컬럼 수)

In [28]:
max_length = joblib.load("./model/bank_app_max_length.joblib")
print(max_length)

302


In [29]:
X_padded = pad_sequences(x, maxlen=max_length, padding='post')
print(X_padded[1])

[ 216  717  370 1524   39   44    8  148 1025  724   27 1033   31   43
   14   51  117    3    6    9    4  861  117    9  414  224  112    9
  164  409 2261 1336   20    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [30]:
len(X_padded[1])

302

In [31]:
y = test_data['is_good']
y

0       1
1       1
2       0
3       0
4       1
       ..
9529    0
9530    1
9531    1
9532    1
9533    0
Name: is_good, Length: 9534, dtype: int64

# 모델 불러와서 예측하고 결과 비교하기

In [32]:
birnn_best = load_model("./model/bank_app_review_birnn.keras")
cnn_lstm_best = load_model("./model/bank_app_review_lstm_cnn.keras")
attn_best = load_model("./model/bank_app_review_attn_model.keras")

In [33]:
birnn_pred = birnn_best.predict(X_padded)
cnn_latm_pred = cnn_lstm_best.predict(X_padded)
attn_pred = attn_best.predict(X_padded)

298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step
298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step
298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step


In [34]:
birnn_pred = pd.DataFrame(birnn_pred)
cnn_lstm_pred = pd.DataFrame(cnn_latm_pred)
attn_pred = pd.DataFrame(attn_pred)

In [35]:
y = pd.DataFrame(y)

In [36]:
birnn_result = y.join(birnn_pred)
cnn_lstm_result = y.join(cnn_lstm_pred)
attn_pred_result = y.join(attn_pred)

In [37]:
birnn_result.loc[:, 0] = birnn_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
cnn_lstm_result.loc[:, 0] = cnn_lstm_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)
attn_pred_result.loc[:, 0] = attn_pred_result.loc[:, 0].apply(lambda x: 1 if x > 0.5 else 0)

In [38]:
birnn_result

,is_good,0
0,1,1.0
1,1,1.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


In [39]:
cnn_lstm_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


In [40]:
attn_pred_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


In [41]:
from sklearn.metrics import classification_report

In [42]:
print(classification_report(birnn_result['is_good'], birnn_result[0]))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      3862
           1       0.92      0.89      0.91      5672

    accuracy                           0.89      9534
   macro avg       0.88      0.89      0.89      9534
weighted avg       0.89      0.89      0.89      9534



In [43]:
print(classification_report(cnn_lstm_result['is_good'], cnn_lstm_result[0]))

              precision    recall  f1-score   support

           0       0.85      0.90      0.88      3862
           1       0.93      0.89      0.91      5672

    accuracy                           0.90      9534
   macro avg       0.89      0.90      0.89      9534
weighted avg       0.90      0.90      0.90      9534



In [44]:
print(classification_report(attn_pred_result['is_good'], attn_pred_result[0]))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      3862
           1       0.92      0.89      0.91      5672

    accuracy                           0.89      9534
   macro avg       0.89      0.89      0.89      9534
weighted avg       0.89      0.89      0.89      9534



In [45]:
attn_pred_result

,is_good,0
0,1,1.0
1,1,0.0
2,0,0.0
3,0,1.0
4,1,1.0
...,...,...
9529,0,0.0
9530,1,1.0
9531,1,1.0
9532,1,1.0


# evaluate

In [46]:
%%time
birnn_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.8922 - auc: 0.9474 - loss: 0.2820
CPU times: user 20.4 s, sys: 5 s, total: 25.4 s
Wall time: 5.98 s


[0.27986541390419006, 0.8906020522117615, 0.9484494924545288]

In [47]:
%%time
cnn_lstm_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8996 - auc: 0.9525 - loss: 0.2726
CPU times: user 20.7 s, sys: 2.36 s, total: 23 s
Wall time: 6.34 s


[0.2722569704055786, 0.8961610794067383, 0.9524178504943848]

In [48]:
%%time
attn_best.evaluate(X_padded, test_data['is_good'])

298/298 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.8954 - auc: 0.9506 - loss: 0.2806
CPU times: user 20.5 s, sys: 2.3 s, total: 22.8 s
Wall time: 6.35 s


[0.2805216610431671, 0.8934340476989746, 0.9500241279602051]